# Associate Refseq and Genebank gene names

This code reads a Refseq .gff file that contains both Refseq and Genebank gene names. Output is a list of Refseq gene names and the corresponding 
Genbank gene name.

The .gff file is read into a dictionary: Refseq gene name -> Genebank gene name.
1. If the line starts with "#", then skip to the next line.
2. If the line contains the the string "locus_tag=CPHY_RS[0-9]+", then match "CPHY_RS[0-9]+" and set it to the variable "refseq".
3. If the line contains the string "old_locus_tag=Cphy_[0-9]+", then match "Cphy_[0-9]+" and set it to the variable "genbank". Otherwise, set "genbnk" to "N/A".
4. Add a new key-value pair to the dictionary "gene_names" where the key is "refseq" and the value is "genbank".

Output: TSV file with Refseq gene name, Genbank gene name, Gene annotation

## Setup

In [77]:
# libraries
import re  # regular expressions

In [78]:
# File IO

file_gff = "/home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/Genome_Files/2025.11_Refseq/GCF_000018685.1_ASM1868v1/GCF_000018685.1_ASM1868v1_genomic.gff";
names_out = "/home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/Genome_Files/2025.11_Refseq/gene-names_Refseq_Genbank.tsv";

## Functions

In [79]:
def read_gff(file_path):
    """
    Reads a file line-by-line, extracts 'refseq' (locus_tag) and 
    'genbank' (old_locus_tag) identifiers, and maps them in a dictionary.
    
    Args:
        file_path (str): The path to the input gff file.
        
    Returns:
        dict: A dictionary mapping refseq IDs to GenBank IDs.
    """
    gff_names = {}  # initialize empty dictionary
    
    # Convert regex to pattern objects (using capturing groups)
    refseq_pattern = re.compile(r'locus_tag=(CPHY_RS\d+)')
    genbank_pattern = re.compile(r'old_locus_tag=(Cphy_[R\d]+)')
    gene_line = re.compile(r"ID=gene-CPHY_RS\d+;")

    print(f"Processing file: {file_path}")

    try:
        with open(file_path, 'r') as file:
            for line in file:
                line = line.strip() # remove leading/trailing whitespace and newline
                
                # Skip lines starting with "#" or empty lines
                if line.startswith("#") or not line:
                    continue

                # skip lines that do not refer to "gene" features
                gene_line_match = gene_line.search(line)
                if not gene_line_match:
                    continue
                
                # Reset Refseq for each line 
                refseq = None
                genbank = None

                # Check for locus_tag (refseq)
                refseq_match = refseq_pattern.search(line)
                if refseq_match:
                    refseq = refseq_match.group(1)
                
                # Check for old_locus_tag (genbank)
                genbank_match = genbank_pattern.search(line)
                if genbank_match:
                    genbank = genbank_match.group(1)
                else:
                    pass # move to next line of code

                if refseq not in gff_names:
                    gff_names[refseq] = [genbank]
                
    except FileNotFoundError:
        print(f"Error: Input file not found at {file_path}")
    except Exception as e:
        print(f"An unexpected error occurred during file processing: {e}")
        
    return gff_names

In [ ]:
def add_annotation(file_path, gff_names):
    """
    This function takes two arguments: the path to the file "gff_file" and a dictionary with key = refseq_name and value = genbank_name.

    Define two strings as pattern matches: gene_name_pattern is "locus_tag=(CPHY_RS\d+)", gene_product_pattern is "product=([\w\s]+)".

    Open a file handle to "gff_file" and read the file line-by-line. 

    If the line in gff_file matches both gene_name_pattern and gene_product_pattern, then set gene_name = gene_name_pattern.group(1)
    and product = gene_product_pattern.group(1)

    Read through the dictionary gff_names. If the key in the gff_names dictionary matches gene_name, then add product as 
    the second element in the list pointed to by that key in gff_names.

    Once all the lines in gff_file have been read, return a dictionary gff_names where each key points to a list 
    of two items: genbank_name and annotation. 
    """

    # Define pattern matches
    gene_name_pattern = re.compile(r"locus_tag=(CPHY_RS\d+)")
    gene_product_pattern = re.compile(r"product=([^;]+)")

    try:
        # Open a file handle to "gff_file" and read line-by-line
        with open(file_path, 'r') as f:
            for line in f:
                
               # Attempt to match both patterns in the line
                name_match = gene_name_pattern.search(line)
                product_match = gene_product_pattern.search(line)
                
                # Check if both matches were possible
                if name_match and product_match:
                    
                    # Set gene_name and product by extracting the captured group (group 1)
                    gene_name = name_match.group(1)
                    product = product_match.group(1).strip() # .strip() cleans up extra whitespace
                    
                    # If the key exists in the dictionary:
                    if gene_name in gff_names:
                        if len(gff_names[gene_name]) == 1:
                            gff_names[gene_name].append(product)
                        
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        gff_names_annotation = gff_names
        return gff_names_annotation 
    
    # Once all lines have been read, return the updated dictionary
    gff_names_annotation = gff_names
    return gff_names_annotation


<>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/tmp/ipykernel_27276/2720001895.py:5: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  Define two strings as pattern matches: gene_name_pattern is "locus_tag=(CPHY_RS\d+)", gene_product_pattern is "product=([\w\s]+)".


# Main

In [81]:
# make dictionary of refseq_name -> genbank_name
gff_names = read_gff(file_gff)

# add gene annotations
gff_names_annotation = add_annotation(file_gff, gff_names)

# count number of refseq names found
num_keys = len(gff_names_annotation)
print(f"Total refseq names found: {num_keys}")

# count number of genebank names found
target_string = "Cphy_"
count = 0

# Iterate through values of the dictionary
for key, value_list in gff_names_annotation.items():
    genbank_name = value_list[0]
    if target_string in str(genbank_name):
        count += 1

print(f"{count} Genbank genes are associated with Refseq names")

# print out file of Refseq names, Genbank names, and annotations
with open(names_out, 'w') as file_object:
    print("This file associates Refseq and Genbank names for C.phytofermentans genome using the GFF file for Refseq GCF_000018685.1_ASM1868v1. It was generated using gene-names_Refseq-Genbank.ipynb.\n", file = file_object)
    print("Refseq_name\tGenbank_name\tAnnotation", file = file_object)

    for key, item_list in gff_names.items(): #items() loops through key-value pairs
        output_elements = [str(key)] 
        string_items = [str(item) for item in item_list]
        output_elements.extend(string_items)
        print('\t'.join(output_elements), file = file_object)
       
              

Processing file: /home/tolonen/Drives/Genoscope-googledrive/Lab_Projects/Genome_Cphy/Genome_Files/2025.11_Refseq/GCF_000018685.1_ASM1868v1/GCF_000018685.1_ASM1868v1_genomic.gff
Total refseq names found: 4134
3973 Genbank genes are associated with Refseq names
